# D3 · Experiment 4 · band-averaged graph (GCN)

Dataset 3: band-averaged adjacency of the four bands, nodes = [log, relative] source PSD of all four bands, GCN; PLV and |ciPLV| (the Tier 1 Exp 4 design, methodology log entry 49).

**Runs only this experiment** (`configs/experiments/d3/d3_exp4_band_mean_fixedhp.yaml`): every predefined variant and setting, with subject-level cross-validation, evaluation and checkpointing. It never starts another experiment.

- **2 variant(s):** `plv`, `ciplv`
- **Settings:** within_D3 (cross-dataset only if the DESIGN.md §7 gate passes)
- **Protocol (from the config, unchanged):** 10 repeats × 5-fold subject-level outer CV; fixed hyperparameters, no hyperparameter search and no inner CV: one fit per outer training fold (protocol `gnn_fixed_hparams_v1`); 60 epochs.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Progress** is shown as one updating line: completed outer folds of the whole experiment
(previously completed folds recovered from Drive count as done immediately), current variant,
setting, repeat and fold, inner-CV trial (if any), training epoch, and the counts of finished /
resumed / failed / skipped units. An ETA is shown only for the current variant and setting, from folds
computed in this session. Large GNN experiments take several Colab sessions: just *Run all* again after each disconnect.

Results are saved under `<outputs>/experiments_D3/d3_exp4_band_mean_fixedhp/` on Drive: per-fold/repeat checkpoints,
window and subject predictions, metrics per repeat, selected hyperparameters, summaries, manifests,
runtime and peak memory per unit (`_run_status.csv`), and a report folder (results table +
**single-experiment diagnostic figures**). The comparative paper figures across experiments are made
separately (`03_comparative_figures.ipynb`).

Requires the graphs from `d3/D3_01_download_and_build_graphs.ipynb`. Dataset 3 (OpenNeuro ds004504, AD vs HC): graph runs `graphsD3_v4_*`, results in `experiments_D3/`, never mixed with D1/D2.

In [ ]:
# ===== Dataset 3 setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_D3_CODE = False   # leave False. True only if you were asked to switch D3 to newer code
# Dataset 3 has its own code pin, <outputs>/D3_CODE_COMMIT.txt (all D3 notebooks run that commit);
# the D1/D2 pin CODE_COMMIT.txt is only reported, never read for checkout or changed here.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'D3']
     + (['--update-code'] if UPDATE_D3_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])

## 0. This experiment's code pin
This experiment runs the commit recorded in its own folder (`experiments_D3/d3_exp4_band_mean_fixedhp/CODE_COMMIT.txt`, set on the
first run). The family pin used by the setup cell, the graph run and every other experiment's results stay
unchanged. Leave `UPDATE_EXPERIMENT_CODE = False`; `True` moves only this experiment's pin to the newest code.

In [ ]:
UPDATE_EXPERIMENT_CODE = False
import subprocess, sys
from pathlib import Path
from adgraph.colab import resolve_code_pin

EXP_DIR = Path(OUT) / 'experiments_D3' / 'd3_exp4_band_mean_fixedhp'
head = subprocess.check_output(['git', '-C', '/content/repo', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
has_work = EXP_DIR.exists() and any(EXP_DIR.rglob('*.done.json'))
EXP_COMMIT, _ = resolve_code_pin(EXP_DIR / 'CODE_COMMIT.txt', head, update=UPDATE_EXPERIMENT_CODE, has_work=has_work)
_run(['git', '-C', '/content/repo', 'checkout', '-q', EXP_COMMIT])
if not Path('/content/repo', 'configs/experiments/d3/d3_exp4_band_mean_fixedhp.yaml').exists():
    raise RuntimeError(f'pin {EXP_COMMIT[:10]} lacks configs/experiments/d3/d3_exp4_band_mean_fixedhp.yaml: set UPDATE_EXPERIMENT_CODE = True and run again')
for m in [m for m in sys.modules if m == 'adgraph' or m.startswith('adgraph.')]:
    del sys.modules[m]                     # this notebook's own imports now come from this experiment's commit
from adgraph.colab import sh
print('d3_exp4_band_mean_fixedhp code commit:', EXP_COMMIT)
for pin in ('D3_CODE_COMMIT.txt', 'CODE_COMMIT.txt'):
    if (Path(OUT) / pin).exists():
        print(f'{pin} (not used here, unchanged):', (Path(OUT) / pin).read_text().split()[0])

## 1. This experiment: progress so far (from the checkpoints on Drive)

In [ ]:
CONFIG = 'configs/experiments/d3/d3_exp4_band_mean_fixedhp.yaml'   # this notebook's experiment
EXPERIMENT = 'd3_exp4_band_mean_fixedhp'
import torch
print('runtime:', ('GPU ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('NOTE: GNN experiment on CPU. It works, but a GPU runtime (Runtime > Change runtime type > T4 GPU) '
          'is much faster. Interrupt now if you want to switch; nothing is lost.')
sh(f"python scripts/experiment.py status --config {CONFIG}")

## 2. Run (or resume) this experiment
Saves every completed fold to Drive as it goes. Units that fail are recorded and the others continue; failures are reported at the end of section 3 (after the results are shown) and stop *Run all* with an error.

In [ ]:
RUN_ERROR = None
try:
    sh(f"python scripts/experiment.py run --config {CONFIG}")
except RuntimeError as exc:
    RUN_ERROR = exc
    print('\nSome units failed or are incompatible; details after the results below.')

## 3. Results of this experiment (saved outputs only)
Subject-level metrics: mean ± SD over CV repeats, subject-bootstrap 95 % CI. Runtime and peak memory per variant and setting. Diagnostic figures of this experiment only; not the comparative paper figures. Nothing is selected or ranked here.

In [ ]:
sh(f"python scripts/experiment.py report --config {CONFIG}")
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
exp_dir = Path(OUT) / 'experiments_D3' / EXPERIMENT
report = sorted((exp_dir / '_reports').glob('report_*'))[-1]
print('report folder on Drive:', report)
pd.set_option('display.max_rows', 500, 'display.width', 250)
table = pd.read_csv(report / 'results_summary.csv')
display(table) if len(table) else print('no completed results yet')
status = pd.read_csv(exp_dir / '_run_status.csv') if (exp_dir / '_run_status.csv').exists() else pd.DataFrame()
if len(status):
    print('runtime and peak memory of the last run (seconds, MB):')
    display(status[['variant', 'setting', 'state', 'seconds', 'mem_peak_mb', 'message']])
for png in sorted((report / 'figures').glob('D[12]_*.png')):
    print(png.name)
    display(Image(filename=str(png)))
print('all figures:', sorted(p.name for p in (report / 'figures').glob('*.png')))
bad = status[status['state'].isin(['failed', 'incompatible'])] if len(status) else status
if RUN_ERROR is not None or len(bad):
    raise RuntimeError(f"{len(bad)} unit(s) of {EXPERIMENT} did not complete (see the table above and "
                       f"{exp_dir / '_run_status.csv'}); completed units are saved. Fix the cause, then Run all "
                       "again: only the incomplete units are recomputed.")